# Stress drop estimation using inter-station phase coherence

This code is to estimate the earthquake stress drop using inter-station phase coherence method following: https://doi.org/10.1093/gji/ggy429 (Hawthorne, 2019).

The notebook has two steps, and each step is one cell:

1. Signal processing
2. Phase coherence and bootstrap

# 1. Signal processing

We want to extract the apparent source time function (ASTF) from the seismogram, by using the empirical green function (EGF) method. The two earthquakes are: 
1. the target earthquake, the bigger one, the one we estimate the diameter;
2. the reference earthquake, the smaller earthquake that we use to remove the Green's function.

This step runs the function `data_processing` (data_processing.py), which does:
1. Data preparation and pre-processing of the target and the reference earthquake: catalog, seismograms, detrend, taper, filter, remove instrument response, merge, onset detection (ray tracing and STA/LTA), and signal to noise ratio (SNR).
2. Onset pick time correction by cross-correlation.
3. Final signal selection by the SNR and cross-correlation criteria, and the plot of the signal comparison (target and reference earthquake) at the recorded stations.
4. The final seismograms are saved in `fdir` in the MSEED format (the picks and the SNR are saved in a text file, `_picks.csv`), they are the input of the phase coherence.

In [ ]:
import obspy
import numpy as np
import matplotlib.pyplot as plt
from importlib import reload
import os

plt.rcParams['font.family'] = 'Arial'

# the function of this step, and the functions that it uses
import signal_preprocessing
import taup_predicted_arrivals1
import cross_correlation_function_w_filter
import data_processing
reload(signal_preprocessing)
reload(taup_predicted_arrivals1)
reload(cross_correlation_function_w_filter)
reload(data_processing)

# pick the earthquake time
event_time = obspy.UTCDateTime('2019/07/07 10:37:45')      # target earthquake
event_time_2 = obspy.UTCDateTime('2019/07/06 19:56:23')    # reference earthquake

# folder and file names of the final seismograms
fdir='EXAMPLE_SEISMOGRAMS'
file_1 = 'target_final.mseed'
file_2 = 'reference_final.mseed'

# process the seismograms, until one step before the phase coherence
st1_use, st_2 = data_processing.data_processing(event_time, event_time_2, fdir, file_1, file_2)


# 2. Phase coherence and bootstrap

This step runs the function `phase_coherence_bootstrap` (phase_coherence_bootstrap.py), which reads the final seismograms of step 1 and does:
1. The inter-station phase coherence of all stations with `phscoh.calcxc`, `calcmvout`, `cpfromr` and `pickffreq`, and the fall-off frequency.
2. The bootstrap: the phase coherence is calculated again 1000 times, each time with a random 80% of the stations, without replacement.
3. The figure with four panels: displacement amplitude of the target earthquake (signal and noise), displacement amplitude of the reference earthquake (signal and noise), phase coherence with the bootstrap interval (the fall-off frequency is printed in the legend), and the histogram of the bootstrap fall-off frequency.

Input parameters are:

fdir, file_1, file_2 : folder and file names of the final seismograms, saved in step 1.

trange : time window segment to extract for spectral analysis in second, the same for all stations and all resamples.

In [ ]:
import obspy
import numpy as np
import matplotlib.pyplot as plt
from importlib import reload
import os

plt.rcParams['font.family'] = 'Arial'

# the function of this step, and the phase coherence functions
import phscoh as phscoh
import phase_coherence_bootstrap
reload(phscoh)
reload(phase_coherence_bootstrap)

# folder and file names of the final seismograms saved in step 1
fdir='EXAMPLE_SEISMOGRAMS'
file_1 = 'target_final.mseed'
file_2 = 'reference_final.mseed'

# phase coherence, bootstrap and the four figures
xc, boot_ff = phase_coherence_bootstrap.phase_coherence_bootstrap(fdir, file_1, file_2, trange=[0, 1.7])
